Required Imports

In [ ]:

!pip install torch transformers accelerate  tenacity

!!pip install torch --extra-index-url https://download.pytorch.org/whl/cu118


!pip install sentence-transformers
!pip install evaluate rouge_score nltk

In [ ]:

!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.36.2 sentencepiece protobuf accelerate datasets

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 126.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 66.1 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 2.0.0
    Uninstalling triton-2.0.0:
      Successfully uninstalled triton-2.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [torchaudio]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.15.2 requires transformers, which is not installed.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 129.9 MB/s eta 0:00:00
   ━━━━

Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import re
import string
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

def clean_text(text):
    """Clean and normalize text data"""
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove special characters and extra whitespace
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Convert to lowercase
    text = text.lower()

    return text

def preprocess_unani_data(filepath):
  
    print("Loading dataset...")
    df = pd.read_csv(filepath)


    print("Cleaning data...")
    df = df.dropna(how='all')
    df = df.fillna('')

    df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

    text_cols = ['unani_herb/formulations', 'botanic_name', 'parts_used',
                'temperature', 'ingredients', 'diseases', 'symptoms_relieved',
                'dosage', 'treatment']

    for col in tqdm(text_cols, desc="Cleaning columns"):
        if col in df.columns:
            df[col] = df[col].apply(clean_text)


    print("Standardizing temperature values...")
    temp_mapping = {
        'hot': ['hot', 'warm'],
        'cold': ['cold', 'cool'],
        'dry': ['dry'],
        'wet': ['wet', 'moist']
    }

    def standardize_temp(temp):
        temp = temp.lower()
        for standard, variants in temp_mapping.items():
            if any(v in temp for v in variants):
                return standard
        return temp

    if 'temperature' in df.columns:
        df['temperature'] = df['temperature'].apply(standardize_temp)


    list_cols = ['diseases', 'symptoms_relieved', 'ingredients']

    for col in tqdm(list_cols, desc="Processing list columns"):
        if col in df.columns:
            df[col] = df[col].apply(lambda x: [i.strip() for i in x.split(',')] if x else [])


    if 'unani_herb/formulations' in df.columns:
        df['herb_id'] = df['unani_herb/formulations'].apply(
            lambda x: re.sub(r'[^a-z0-9]', '', x.lower())[:20]
        )

    print("Preprocessing completed!")
    return df

# Usage
df = preprocess_unani_data('sample_data/5000done.csv')
df.to_csv('preprocessed_unani_data.csv', index=False)

Loading dataset...
Cleaning data...


Cleaning columns: 100%|██████████| 9/9 [00:00<00:00, 26.84it/s]


Standardizing temperature values...


Processing list columns: 100%|██████████| 3/3 [00:00<00:00, 166.42it/s]

Preprocessing completed!


Translated Dataset in Urdu

In [ ]:
from transformers import pipeline
from tqdm import tqdm
import pandas as pd
import time
import torch
import ast
import warnings
warnings.filterwarnings('ignore')

def initialize_translator():
    """Initialize translation pipeline with error handling"""
    try:
        device = 0 if torch.cuda.is_available() else -1
        return pipeline(
            task='translation',
            model='facebook/nllb-200-distilled-600M',
            device=device,
            torch_dtype=torch.float16 if device == 0 else torch.float32
        )
    except Exception as e:
        print(f"Failed to initialize translator: {str(e)}")
        return None

def safe_translate(text, translator):
    """Handle translation with robust error catching"""
    if not text or pd.isna(text):
        return ""

    try:
        result = translator(text, src_lang="eng_Latn", tgt_lang="urd_Arab")
        return result[0]['translation_text']
    except Exception as e:
        print(f"Error translating text (first 50 chars: {str(text)[:50]}...): {str(e)}")
        return ""

def process_column(df, col, translator):
    """Process a single column with progress tracking"""
    tqdm.pandas(desc=f"Translating {col}")

    if isinstance(df[col].iloc[0], list):
        df[f"{col}_ur"] = df[col].progress_apply(
            lambda lst: [safe_translate(str(item), translator) for item in lst] if lst else []
        )
    else:
        df[f"{col}_ur"] = df[col].progress_apply(
            lambda x: safe_translate(str(x), translator) if pd.notna(x) else ""
        )

    return df

def translate_dataset(df, cols_to_translate):
    
    translator = initialize_translator()
    if not translator:
        return df

    for col in cols_to_translate:
        if col not in df.columns:
            print(f"Column {col} not found, skipping")
            continue

        print(f"\nStarting translation of {col}...")
        start_time = time.time()

        df = process_column(df, col, translator)

     
        df.to_csv('unani_translated_partial.csv', index=False)
        print(f"Completed {col} in {time.time()-start_time:.2f} seconds")

    return df

def safe_convert_to_list(x):
   
    try:
        return ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith('[') else []
    except:
        return []

def main():
    try:

        df = pd.read_csv('preprocessed_unani_data.csv')

    
        list_cols = ['diseases', 'symptoms_relieved', 'ingredients']
        for col in list_cols:
            if col in df.columns:
                df[col] = df[col].apply(safe_convert_to_list)


        cols_to_translate = [
            'unani_herb/formulations',
            'diseases',
            'symptoms_relieved',
            'treatment'
        ]

        df = translate_dataset(df, cols_to_translate)

        # Save final result
        df.to_csv('unani_translated_final.csv', index=False)
        print("\nTranslation completed successfully!")

    except Exception as e:
        print(f"\nFatal error in main execution: {str(e)}")

if __name__ == "__main__":
    main()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.11/dist-package

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...

Completed diseases in 1407.69 seconds

Starting translation of symptoms_relieved...


Translating symptoms_relieved:  56%|█████▌    | 2807/5041 [10:40<40:43,  1.09s/it]Token indices sequence length is longer than the specified maximum sequence length for this model (1245 > 1024). Running this sequence through the model will result in indexing errors
Your input_length: 1245 is bigger than 0.9 * max_length: 200. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Translating symptoms_relieved: 100%|██████████| 5041/5041 [19:52<00:00,  4.23it/s]


Completed symptoms_relieved in 1192.27 seconds

Starting translation of treatment...


Translating treatment: 100%|██████████| 5041/5041 [36:21<00:00,  2.31it/s]


Completed treatment in 2181.73 seconds

Translation completed successfully!


NEO4J Construction

In [ ]:
from neo4j import GraphDatabase
import pandas as pd
from tqdm import tqdm


URI = ""
USER = ""
PASSWORD = ""

def clean_data(value, max_length=65535):
    """Helper function to clean string values with length limit"""
    if pd.isna(value) or value == "":
        return ""
    return str(value).strip()[:max_length]

def split_values(value, max_length=65535):
    """Helper function to split comma-separated values with length limit"""
    if pd.isna(value) or value == "":
        return []
    return [v.strip()[:max_length] for v in str(value).split(",") if v.strip()]

def create_constraints(tx):
    """Create all necessary constraints in Neo4j"""
    constraints = [
       
        "CREATE CONSTRAINT unani_formulation_unique IF NOT EXISTS FOR (f:UnaniFormulation) REQUIRE (f.name, f.language) IS UNIQUE",
        "CREATE CONSTRAINT botanical_name_unique IF NOT EXISTS FOR (b:BotanicalName) REQUIRE (b.name, b.language) IS UNIQUE",
        "CREATE CONSTRAINT plant_part_unique IF NOT EXISTS FOR (p:PlantPart) REQUIRE (p.name, p.language) IS UNIQUE",
        "CREATE CONSTRAINT temperament_unique IF NOT EXISTS FOR (t:Temperament) REQUIRE (t.name, t.language) IS UNIQUE",
        "CREATE CONSTRAINT ingredient_unique IF NOT EXISTS FOR (i:Ingredient) REQUIRE (i.name, i.language) IS UNIQUE",
        "CREATE CONSTRAINT disease_unique IF NOT EXISTS FOR (d:Disease) REQUIRE (d.name, d.language) IS UNIQUE",
        "CREATE CONSTRAINT symptom_unique IF NOT EXISTS FOR (s:Symptom) REQUIRE (s.name, s.language) IS UNIQUE",
        "CREATE CONSTRAINT dosage_unique IF NOT EXISTS FOR (d:Dosage) REQUIRE (d.description, d.language) IS UNIQUE",
        "CREATE CONSTRAINT treatment_unique IF NOT EXISTS FOR (t:Treatment) REQUIRE (t.description, t.language) IS UNIQUE",
    ]

    for constraint in constraints:
        try:
            tx.run(constraint)
        except Exception as e:
            print(f"Error creating constraint: {constraint}")
            print(e)

def create_indexes(tx):
    """Create all necessary indexes in Neo4j"""
    indexes = [
      
        "CREATE FULLTEXT INDEX unaniFormulationSearch IF NOT EXISTS FOR (f:UnaniFormulation) ON EACH [f.name]",
        "CREATE FULLTEXT INDEX botanicalNameSearch IF NOT EXISTS FOR (b:BotanicalName) ON EACH [b.name]",
        "CREATE FULLTEXT INDEX diseaseSearch IF NOT EXISTS FOR (d:Disease) ON EACH [d.name]",
        "CREATE FULLTEXT INDEX symptomSearch IF NOT EXISTS FOR (s:Symptom) ON EACH [s.name]",

        
        "CREATE INDEX unani_formulation_lang IF NOT EXISTS FOR (f:UnaniFormulation) ON f.language",
        "CREATE INDEX botanical_name_lang IF NOT EXISTS FOR (b:BotanicalName) ON b.language",
    ]

    for index in indexes:
        try:
            tx.run(index)
        except Exception as e:
            print(f"Error creating index: {index}")
            print(e)

def create_nodes_and_relationships(tx, row):
    """Create all nodes and relationships for a given row"""

    tx.run("""
        MERGE (form:UnaniFormulation {name: $name, language: $lang})
    """, name=clean_data(row["unani_herb__formulations"]), lang="English")

  
    tx.run("""
        MERGE (bot:BotanicalName {name: $name, language: $lang})
    """, name=clean_data(row["botanic_name"]), lang="English")

    # Plant Part (English)
    tx.run("""
        MERGE (part:PlantPart {name: $name, language: $lang})
    """, name=clean_data(row["parts_used"]), lang="English")

    # Temperament (English)
    tx.run("""
        MERGE (temp:Temperament {name: $name, language: $lang})
    """, name=clean_data(row["temperament"]), lang="English")

    # Ingredients (English)
    for ing in split_values(row["ingredients"]):
        tx.run("""
            MERGE (ing:Ingredient {name: $name, language: $lang})
        """, name=ing, lang="English")

    # Diseases (English)
    for dis in split_values(row["diseases"]):
        tx.run("""
            MERGE (dis:Disease {name: $name, language: $lang})
        """, name=dis, lang="English")

    # Symptoms (English)
    for sym in split_values(row["symptoms_relieved"]):
        tx.run("""
            MERGE (sym:Symptom {name: $name, language: $lang})
        """, name=sym, lang="English")

    # Dosage (English)
    tx.run("""
        MERGE (dos:Dosage {description: $desc, language: $lang})
    """, desc=clean_data(row["dosage"]), lang="English")

    # Treatment (English)
    tx.run("""
        MERGE (treat:Treatment {description: $desc, language: $lang})
    """, desc=clean_data(row["treatment"]), lang="English")

    # Create Urdu nodes
    # Unani Formulation (Urdu)
    tx.run("""
        MERGE (form:UnaniFormulation {name: $name, language: $lang})
    """, name=clean_data(row["unani_herb__formulations_ur"]), lang="Urdu")

    # Botanical Name (Urdu)
    tx.run("""
        MERGE (bot:BotanicalName {name: $name, language: $lang})
    """, name=clean_data(row["botanic_name_ur"]), lang="Urdu")

    # Plant Part (Urdu)
    tx.run("""
        MERGE (part:PlantPart {name: $name, language: $lang})
    """, name=clean_data(row["parts_used_ur"]), lang="Urdu")

    # Temperament (Urdu)
    tx.run("""
        MERGE (temp:Temperament {name: $name, language: $lang})
    """, name=clean_data(row["temperament_ur"]), lang="Urdu")

    # Ingredients (Urdu)
    for ing in split_values(row["ingredients_ur"]):
        tx.run("""
            MERGE (ing:Ingredient {name: $name, language: $lang})
        """, name=ing, lang="Urdu")

    # Diseases (Urdu)
    for dis in split_values(row["diseases_ur"]):
        tx.run("""
            MERGE (dis:Disease {name: $name, language: $lang})
        """, name=dis, lang="Urdu")

    # Symptoms (Urdu)
    for sym in split_values(row["symptoms_relieved_ur"]):
        tx.run("""
            MERGE (sym:Symptom {name: $name, language: $lang})
        """, name=sym, lang="Urdu")

    # Dosage (Urdu)
    tx.run("""
        MERGE (dos:Dosage {description: $desc, language: $lang})
    """, desc=clean_data(row["dosage_ur"]), lang="Urdu")

    # Treatment (Urdu)
    tx.run("""
        MERGE (treat:Treatment {description: $desc, language: $lang})
    """, desc=clean_data(row["treatment_ur"]), lang="Urdu")

    
    # Formulation to Botanical Name
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (bot:BotanicalName {name: $bot_name, language: $lang})
        MERGE (form)-[:HAS_BOTANICAL_NAME]->(bot)
    """, form_name=clean_data(row["unani_herb__formulations"]),
          bot_name=clean_data(row["botanic_name"]),
          lang="English")

    # Formulation to Plant Part
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (part:PlantPart {name: $part_name, language: $lang})
        MERGE (form)-[:USES_PLANT_PART]->(part)
    """, form_name=clean_data(row["unani_herb__formulations"]),
          part_name=clean_data(row["parts_used"]),
          lang="English")

    # Formulation to Temperament
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (temp:Temperament {name: $temp_name, language: $lang})
        MERGE (form)-[:HAS_TEMPERAMENT]->(temp)
    """, form_name=clean_data(row["unani_herb__formulations"]),
          temp_name=clean_data(row["temperament"]),
          lang="English")

    # Formulation to Ingredients
    for ing in split_values(row["ingredients"]):
        tx.run("""
            MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
            MATCH (ing:Ingredient {name: $ing_name, language: $lang})
            MERGE (form)-[:CONTAINS_INGREDIENT]->(ing)
        """, form_name=clean_data(row["unani_herb__formulations"]),
              ing_name=ing,
              lang="English")

    # Formulation to Diseases
    for dis in split_values(row["diseases"]):
        tx.run("""
            MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
            MATCH (dis:Disease {name: $dis_name, language: $lang})
            MERGE (form)-[:TREATS_DISEASE]->(dis)
        """, form_name=clean_data(row["unani_herb__formulations"]),
              dis_name=dis,
              lang="English")

    # Formulation to Symptoms
    for sym in split_values(row["symptoms_relieved"]):
        tx.run("""
            MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
            MATCH (sym:Symptom {name: $sym_name, language: $lang})
            MERGE (form)-[:RELIEVES_SYMPTOM]->(sym)
        """, form_name=clean_data(row["unani_herb__formulations"]),
              sym_name=sym,
              lang="English")

    # Formulation to Dosage
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (dos:Dosage {description: $dos_desc, language: $lang})
        MERGE (form)-[:HAS_DOSAGE]->(dos)
    """, form_name=clean_data(row["unani_herb__formulations"]),
          dos_desc=clean_data(row["dosage"]),
          lang="English")

    # Formulation to Treatment
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (treat:Treatment {description: $treat_desc, language: $lang})
        MERGE (form)-[:HAS_TREATMENT]->(treat)
    """, form_name=clean_data(row["unani_herb__formulations"]),
          treat_desc=clean_data(row["treatment"]),
          lang="English")

  
    # Formulation to Botanical Name
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (bot:BotanicalName {name: $bot_name, language: $lang})
        MERGE (form)-[:HAS_BOTANICAL_NAME]->(bot)
    """, form_name=clean_data(row["unani_herb__formulations_ur"]),
          bot_name=clean_data(row["botanic_name_ur"]),
          lang="Urdu")

    # Formulation to Plant Part
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (part:PlantPart {name: $part_name, language: $lang})
        MERGE (form)-[:USES_PLANT_PART]->(part)
    """, form_name=clean_data(row["unani_herb__formulations_ur"]),
          part_name=clean_data(row["parts_used_ur"]),
          lang="Urdu")

    # Formulation to Temperament
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (temp:Temperament {name: $temp_name, language: $lang})
        MERGE (form)-[:HAS_TEMPERAMENT]->(temp)
    """, form_name=clean_data(row["unani_herb__formulations_ur"]),
          temp_name=clean_data(row["temperament_ur"]),
          lang="Urdu")

    # Formulation to Ingredients
    for ing in split_values(row["ingredients_ur"]):
        tx.run("""
            MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
            MATCH (ing:Ingredient {name: $ing_name, language: $lang})
            MERGE (form)-[:CONTAINS_INGREDIENT]->(ing)
        """, form_name=clean_data(row["unani_herb__formulations_ur"]),
              ing_name=ing,
              lang="Urdu")

    # Formulation to Diseases
    for dis in split_values(row["diseases_ur"]):
        tx.run("""
            MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
            MATCH (dis:Disease {name: $dis_name, language: $lang})
            MERGE (form)-[:TREATS_DISEASE]->(dis)
        """, form_name=clean_data(row["unani_herb__formulations_ur"]),
              dis_name=dis,
              lang="Urdu")

    # Formulation to Symptoms
    for sym in split_values(row["symptoms_relieved_ur"]):
        tx.run("""
            MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
            MATCH (sym:Symptom {name: $sym_name, language: $lang})
            MERGE (form)-[:RELIEVES_SYMPTOM]->(sym)
        """, form_name=clean_data(row["unani_herb__formulations_ur"]),
              sym_name=sym,
              lang="Urdu")

    # Formulation to Dosage
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (dos:Dosage {description: $dos_desc, language: $lang})
        MERGE (form)-[:HAS_DOSAGE]->(dos)
    """, form_name=clean_data(row["unani_herb__formulations_ur"]),
          dos_desc=clean_data(row["dosage_ur"]),
          lang="Urdu")

    # Formulation to Treatment
    tx.run("""
        MATCH (form:UnaniFormulation {name: $form_name, language: $lang})
        MATCH (treat:Treatment {description: $treat_desc, language: $lang})
        MERGE (form)-[:HAS_TREATMENT]->(treat)
    """, form_name=clean_data(row["unani_herb__formulations_ur"]),
          treat_desc=clean_data(row["treatment_ur"]),
          lang="Urdu")

    # Formulation
    tx.run("""
        MATCH (form_en:UnaniFormulation {name: $form_en, language: "English"})
        MATCH (form_ur:UnaniFormulation {name: $form_ur, language: "Urdu"})
        MERGE (form_en)-[:TRANSLATES_TO]->(form_ur)
        MERGE (form_ur)-[:TRANSLATES_TO]->(form_en)
    """, form_en=clean_data(row["unani_herb__formulations"]),
          form_ur=clean_data(row["unani_herb__formulations_ur"]))

    # Botanical Name
    tx.run("""
        MATCH (bot_en:BotanicalName {name: $bot_en, language: "English"})
        MATCH (bot_ur:BotanicalName {name: $bot_ur, language: "Urdu"})
        MERGE (bot_en)-[:TRANSLATES_TO]->(bot_ur)
        MERGE (bot_ur)-[:TRANSLATES_TO]->(bot_en)
    """, bot_en=clean_data(row["botanic_name"]),
          bot_ur=clean_data(row["botanic_name_ur"]))

    # Plant Part
    tx.run("""
        MATCH (part_en:PlantPart {name: $part_en, language: "English"})
        MATCH (part_ur:PlantPart {name: $part_ur, language: "Urdu"})
        MERGE (part_en)-[:TRANSLATES_TO]->(part_ur)
        MERGE (part_ur)-[:TRANSLATES_TO]->(part_en)
    """, part_en=clean_data(row["parts_used"]),
          part_ur=clean_data(row["parts_used_ur"]))

    # Temperament
    tx.run("""
        MATCH (temp_en:Temperament {name: $temp_en, language: "English"})
        MATCH (temp_ur:Temperament {name: $temp_ur, language: "Urdu"})
        MERGE (temp_en)-[:TRANSLATES_TO]->(temp_ur)
        MERGE (temp_ur)-[:TRANSLATES_TO]->(temp_en)
    """, temp_en=clean_data(row["temperament"]),
          temp_ur=clean_data(row["temperament_ur"]))

    # Ingredients
    ingredients_en = split_values(row["ingredients"])
    ingredients_ur = split_values(row["ingredients_ur"])
    for ing_en, ing_ur in zip(ingredients_en, ingredients_ur):
        tx.run("""
            MATCH (ing_en:Ingredient {name: $ing_en, language: "English"})
            MATCH (ing_ur:Ingredient {name: $ing_ur, language: "Urdu"})
            MERGE (ing_en)-[:TRANSLATES_TO]->(ing_ur)
            MERGE (ing_ur)-[:TRANSLATES_TO]->(ing_en)
        """, ing_en=ing_en, ing_ur=ing_ur)

    # Diseases
    diseases_en = split_values(row["diseases"])
    diseases_ur = split_values(row["diseases_ur"])
    for dis_en, dis_ur in zip(diseases_en, diseases_ur):
        tx.run("""
            MATCH (dis_en:Disease {name: $dis_en, language: "English"})
            MATCH (dis_ur:Disease {name: $dis_ur, language: "Urdu"})
            MERGE (dis_en)-[:TRANSLATES_TO]->(dis_ur)
            MERGE (dis_ur)-[:TRANSLATES_TO]->(dis_en)
        """, dis_en=dis_en, dis_ur=dis_ur)

    # Symptoms
    symptoms_en = split_values(row["symptoms_relieved"])
    symptoms_ur = split_values(row["symptoms_relieved_ur"])
    for sym_en, sym_ur in zip(symptoms_en, symptoms_ur):
        tx.run("""
            MATCH (sym_en:Symptom {name: $sym_en, language: "English"})
            MATCH (sym_ur:Symptom {name: $sym_ur, language: "Urdu"})
            MERGE (sym_en)-[:TRANSLATES_TO]->(sym_ur)
            MERGE (sym_ur)-[:TRANSLATES_TO]->(sym_en)
        """, sym_en=sym_en, sym_ur=sym_ur)

    # Dosage
    tx.run("""
        MATCH (dos_en:Dosage {description: $dos_en, language: "English"})
        MATCH (dos_ur:Dosage {description: $dos_ur, language: "Urdu"})
        MERGE (dos_en)-[:TRANSLATES_TO]->(dos_ur)
        MERGE (dos_ur)-[:TRANSLATES_TO]->(dos_en)
    """, dos_en=clean_data(row["dosage"]),
          dos_ur=clean_data(row["dosage_ur"]))

    # Treatment
    tx.run("""
        MATCH (treat_en:Treatment {description: $treat_en, language: "English"})
        MATCH (treat_ur:Treatment {description: $treat_ur, language: "Urdu"})
        MERGE (treat_en)-[:TRANSLATES_TO]->(treat_ur)
        MERGE (treat_ur)-[:TRANSLATES_TO]->(treat_en)
    """, treat_en=clean_data(row["treatment"]),
          treat_ur=clean_data(row["treatment_ur"]))

   
    for dis in split_values(row["diseases"]):
        for sym in split_values(row["symptoms_relieved"]):
            tx.run("""
                MATCH (dis:Disease {name: $dis_name, language: "English"})
                MATCH (sym:Symptom {name: $sym_name, language: "English"})
                MERGE (dis)-[:HAS_SYMPTOM]->(sym)
            """, dis_name=dis, sym_name=sym)

    # Connect diseases to their symptoms (Urdu)
    for dis in split_values(row["diseases_ur"]):
        for sym in split_values(row["symptoms_relieved_ur"]):
            tx.run("""
                MATCH (dis:Disease {name: $dis_name, language: "Urdu"})
                MATCH (sym:Symptom {name: $sym_name, language: "Urdu"})
                MERGE (dis)-[:HAS_SYMPTOM]->(sym)
            """, dis_name=dis, sym_name=sym)

    # Connect ingredients to their plant parts (English)
    for ing in split_values(row["ingredients"]):
        tx.run("""
            MATCH (ing:Ingredient {name: $ing_name, language: "English"})
            MATCH (part:PlantPart {name: $part_name, language: "English"})
            MERGE (ing)-[:FOUND_IN_PART]->(part)
        """, ing_name=ing, part_name=clean_data(row["parts_used"]))

    # Connect ingredients to their plant parts (Urdu)
    for ing in split_values(row["ingredients_ur"]):
        tx.run("""
            MATCH (ing:Ingredient {name: $ing_name, language: "Urdu"})
            MATCH (part:PlantPart {name: $part_name, language: "Urdu"})
            MERGE (ing)-[:FOUND_IN_PART]->(part)
        """, ing_name=ing, part_name=clean_data(row["parts_used_ur"]))

def import_data():
    """Main function to import the data"""
    try:
  
        try:
            df = pd.read_csv("data.csv", on_bad_lines='warn')
        except:
     
            df = pd.read_csv("data.csv", error_bad_lines=False)

        df = df.fillna("")


        driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

        with driver.session() as session:
            print("Creating constraints...")
            session.execute_write(create_constraints)

            print("Creating indexes...")
            session.execute_write(create_indexes)

            print(f"Importing {len(df)} rows...")
            for _, row in tqdm(df.iterrows(), total=len(df)):
                try:
                    session.execute_write(create_nodes_and_relationships, row)
                except Exception as e:
                    print(f"Error processing row (continuing anyway): {e}")
                    continue

            print("Data import completed successfully!")

        driver.close()
    except Exception as e:
        print(f"Fatal error in import_data: {e}")
        raise

if __name__ == "__main__":
    import_data()

In [ ]:

!pip install neo4j pandas

import os
from neo4j import GraphDatabase
import pandas as pd


os.environ['NEO4J_URI'] = ''
os.environ['NEO4J_USER'] = ''
os.environ['NEO4J_PASSWORD'] = ''

class Neo4jConnection:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [record for record in result]


try:
    conn = Neo4jConnection(
        os.environ['NEO4J_URI'],
        os.environ['NEO4J_USER'],
        os.environ['NEO4J_PASSWORD']
    )
    print(" Successfully connected to Neo4j!")
except Exception as e:
    print(f" Connection failed: {e}")
    exit()

def get_total_nodes():
    query = """
    MATCH (n)
    RETURN count(n) as total_nodes
    """
    result = conn.query(query)
    return result[0]['total_nodes']

def get_total_relationships():
    query = """
    MATCH ()-[r]->()
    RETURN count(r) as total_relationships
    """
    result = conn.query(query)
    return result[0]['total_relationships']

def get_node_types():
    query = """
    MATCH (n)
    RETURN labels(n) as labels, count(n) as count
    ORDER BY count DESC
    """
    result = conn.query(query)
    return result

def get_relationship_types():
    query = """
    MATCH ()-[r]->()
    RETURN type(r) as relationship_type,
           count(r) as count,
           collect(DISTINCT {
             start_node_labels: labels(startNode(r)),
             end_node_labels: labels(endNode(r))
           })[0..5] as sample_connections
    ORDER BY count DESC
    """
    result = conn.query(query)
    return result


def get_detailed_relationships():
    query = """
    MATCH (start)-[r]->(end)
    WITH type(r) as rel_type,
         labels(start) as start_labels,
         labels(end) as end_labels,
         count(r) as count
    RETURN rel_type,
           start_labels,
           end_labels,
           count
    ORDER BY count DESC
    LIMIT 20
    """
    result = conn.query(query)
    return result


print("\n" + "="*60)
print("NEO4J DATABASE ANALYSIS")
print("="*60)


total_nodes = get_total_nodes()
total_relationships = get_total_relationships()

print(f"\n TOTAL COUNTS:")
print(f"   Nodes: {total_nodes:,}")
print(f"   Relationships: {total_relationships:,}")


print(f"\n  NODE TYPES:")
node_types = get_node_types()
for record in node_types:
    labels = record['labels']
    count = record['count']
    print(f"   {labels}: {count:,} nodes")


print(f"\n RELATIONSHIP TYPES:")
rel_types = get_relationship_types()
for record in rel_types:
    rel_type = record['relationship_type']
    count = record['count']
    samples = record['sample_connections']

    print(f"\n   {rel_type}: {count:,} relationships")
    print(f"   Sample connections:")
    for i, sample in enumerate(samples[:3], 1):
        start_labels = sample['start_node_labels']
        end_labels = sample['end_node_labels']
        print(f"     {i}. {start_labels} → {end_labels}")


print(f"\n" + "="*60)
print("DETAILED RELATIONSHIP ANALYSIS")
print("="*60)

detailed_rels = get_detailed_relationships()
for record in detailed_rels:
    rel_type = record['rel_type']
    start_labels = record['start_labels']
    end_labels = record['end_labels']
    count = record['count']

    print(f"\n   {rel_type}:")
    print(f"     From: {start_labels}")
    print(f"     To: {end_labels}")
    print(f"     Count: {count:,}")


conn.close()
print(f"\n Analysis complete! Connection closed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 5.3 MB/s eta 0:00:00
✅ Successfully connected to Neo4j!

NEO4J DATABASE ANALYSIS

📊 TOTAL COUNTS:
   Nodes: 34,725
   Relationships: 114,532

🏷️  NODE TYPES:
   ['UnaniFormulation']: 7,308 nodes
   ['Treatment']: 6,080 nodes
   ['Disease']: 5,836 nodes
   ['Ingredient']: 4,629 nodes
   ['Symptom']: 4,062 nodes
   ['BotanicalName']: 3,588 nodes
   ['Dosage']: 2,626 nodes
   ['PlantPart']: 566 nodes
   ['Temperament']: 30 nodes

🔗 RELATIONSHIP TYPES:

   TRANSLATES_TO: 35,096 relationships
   Sample connections:
     1. ['Treatment'] → ['Treatment']
     2. ['UnaniFormulation'] → ['UnaniFormulation']
     3. ['Ingredient'] → ['Ingredient']

   HAS_TREATMENT: 8,631 relationships
   Sample connections:
     1. ['UnaniFormulation'] → ['Treatment']

   HAS_DOSAGE: 8,604 relationships
   Sample connections:
     1. ['UnaniFormulation'] → ['Dosage']

   TREATS_DISEASE: 8,513 relationships
   Sample connections:
     1. ['UnaniFormulati